# Project Continue — Automatic Backup Controller

This notebook runs versioned backups to Google Drive using `scripts/colab_persistence.py`.

**What it does:** one backup immediately, then a backup every 15 minutes while this Colab runtime is alive. Call `pc_backup_now()` after important changes for an event-triggered backup. It never overwrites an existing snapshot.

**Important limits:** Colab free runtimes can disconnect or stop; a Python thread cannot run after the runtime stops. This is not an always-on scheduler. Keep notebooks in the configured Drive folder, and configure live database/agent-state paths below. A runtime-only agent process cannot be restored—only files/checkpoints can.

Run cells in order. Review the printed manifest and warnings after the first backup.

In [ ]:
# CELL 1 — Mount Drive and configure paths
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, time, threading, importlib.util, traceback

# Change these paths if your live checkout/database/state are elsewhere.
os.environ.setdefault('PROJECT_CONTINUE_REPO', '/content/project-continue')
os.environ.setdefault('PROJECT_CONTINUE_BACKUP_ROOT', '/content/drive/MyDrive/Project-Continue-Backups')
os.environ.setdefault('PROJECT_CONTINUE_NOTEBOOK_DIR', '/content/drive/MyDrive/Project-Continue-Notebooks')
# Set these to the REAL active paths when applicable; do not guess them.
# os.environ['PROJECT_CONTINUE_DB'] = '/content/project-continue/.data/project_continue.sqlite3'
# os.environ['PROJECT_CONTINUE_AGENT_STATE'] = '/content/.project_continue_agent_state.json'
# os.environ['PROJECT_CONTINUE_LOG'] = '/content/project-continue/logs/streamlit.log'

REPO = Path(os.environ['PROJECT_CONTINUE_REPO'])
SCRIPT = REPO / 'scripts' / 'colab_persistence.py'
if not SCRIPT.is_file():
    raise FileNotFoundError(f'{SCRIPT} not found. Clone/open the project repository at {REPO}, or set PROJECT_CONTINUE_REPO to its actual path.')
spec = importlib.util.spec_from_file_location('project_continue_persistence', SCRIPT)
pc = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pc
spec.loader.exec_module(pc)
print('Backup controller loaded:', SCRIPT)
print('Backup destination:', os.environ['PROJECT_CONTINUE_BACKUP_ROOT'])
print('Notebook source folder:', os.environ['PROJECT_CONTINUE_NOTEBOOK_DIR'])

In [ ]:
# CELL 2 — First backup now
def pc_backup_now():
    """Create and verify a new immutable backup; raises on any failure."""
    manifest = pc.save()
    print('BACKUP SUCCESS:', manifest)
    data = json.loads(Path(manifest).read_text(encoding='utf-8'))
    print('Git HEAD:', data['repository'].get('commit'))
    print('Notebook count:', data['notebooks'].get('count'))
    print('Database records:', data.get('database'))
    print('Agent/runtime records:', data.get('runtime_files'))
    return manifest

first_manifest = pc_backup_now()

In [ ]:
# CELL 3 — Start 15-minute backups while this runtime remains alive
PC_BACKUP_INTERVAL_SECONDS = 15 * 60
PC_BACKUP_STOP = threading.Event()
PC_BACKUP_THREAD = None
PC_BACKUP_ERRORS = []

def _pc_backup_worker():
    while not PC_BACKUP_STOP.wait(PC_BACKUP_INTERVAL_SECONDS):
        try:
            pc_backup_now()
        except Exception as exc:
            record = {'time_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'error': repr(exc)}
            PC_BACKUP_ERRORS.append(record)
            print('AUTOMATIC BACKUP FAILED:', record)
            traceback.print_exc()

def pc_start_auto_backup():
    global PC_BACKUP_THREAD
    if PC_BACKUP_THREAD and PC_BACKUP_THREAD.is_alive():
        print('15-minute backup loop is already running.')
        return
    PC_BACKUP_STOP.clear()
    PC_BACKUP_THREAD = threading.Thread(target=_pc_backup_worker, name='ProjectContinueBackup', daemon=True)
    PC_BACKUP_THREAD.start()
    print('15-minute backup loop started. It runs only while this Colab runtime stays alive.')

def pc_stop_auto_backup():
    PC_BACKUP_STOP.set()
    print('Backup loop stop requested.')

pc_start_auto_backup()

In [ ]:
# CELL 4 — Important-change trigger and health status
# Call pc_backup_now() after important code/config/database/checkpoint changes.
def pc_backup_status():
    alive = bool(PC_BACKUP_THREAD and PC_BACKUP_THREAD.is_alive())
    pointer = Path(os.environ['PROJECT_CONTINUE_BACKUP_ROOT']) / 'LATEST_SUCCESSFUL.json'
    latest = json.loads(pointer.read_text()) if pointer.is_file() else None
    return {'loop_alive': alive, 'interval_seconds': PC_BACKUP_INTERVAL_SECONDS,
            'latest_successful': latest, 'recent_errors': PC_BACKUP_ERRORS[-5:]}

print(json.dumps(pc_backup_status(), indent=2))

In [ ]:
# CELL 5 — Restore latest verified backup into an isolated new folder (optional)
# Run only in a NEW runtime when recovery is needed; it does not overwrite live files.
# restore_path = pc.restore_latest()
# print('ISOLATED RESTORE CREATED:', restore_path)